# PGL Auth - Autenticacao e Uso do Proxy OpenAI

**Nome:** Renan Santos Mendes

**Email:** renansantosmendes@gmail.com

**Disciplina:** [NOME_DA_DISCIPLINA]

**Curso:** [NOME_DO_CURSO]

---

## Objetivo

Este notebook demonstra como realizar o cadastro (registro) e a autenticacao (login) de um usuario utilizando o pacote `pgl-auth`, e como utilizar o token gerado para consumir um modelo de linguagem por meio de um proxy compativel com a API da OpenAI.

## Etapas abordadas

1. Instalacao dos pacotes necessarios
2. Registro do usuario (cadastro de senha vinculado a matricula)
3. Login e obtencao do token de autenticacao
4. Visualizacao do token gerado
5. Uso do token para consumir um modelo de linguagem via `langchain-openai`


## 1. Instalacao dos pacotes

A celula abaixo instala os pacotes `pgl-auth` e `langchain-openai` utilizando o `uv` como gerenciador de pacotes, conforme recomendado para o ambiente Colab.

- `pgl-auth`: cliente responsavel pelo registro e login do usuario, retornando um token de autenticacao.
- `langchain-openai`: integracao do LangChain com APIs compativeis com o padrao OpenAI, utilizada posteriormente para consumir o modelo de linguagem.


In [ ]:
!uv pip install pgl-auth langchain-openai

## 2. Registro do usuario

Nesta etapa e criado um cliente `PGLAuthClient` e realizada uma tentativa de registro (`register`) utilizando a matricula (`PGL_REGISTRATION_NUMBER`) e a senha (`PGL_PASSWORD`) armazenadas de forma segura nos *secrets* do Colab (`userdata`).

Caso a matricula ja possua uma senha cadastrada, o servico retorna uma excecao informativa, que e capturada e exibida sem interromper a execucao do notebook. Isso torna a celula segura para ser executada multiplas vezes.


In [ ]:
from google.colab import userdata
from pgl_auth import PGLAuthClient
from langchain_openai import ChatOpenAI

client = PGLAuthClient()
try:
    client.register(
        registration_number=userdata.get("PGL_REGISTRATION_NUMBER"),
        password=userdata.get("PGL_PASSWORD"),
    )
except Exception as e:
    print(e)

## 3. Login e obtencao do token

Apos o registro, o metodo `login` do `PGLAuthClient` e utilizado para autenticar o usuario com a mesma matricula e senha. Se as credenciais estiverem corretas, um token de autenticacao (JWT) e retornado e armazenado na variavel `token`.

Esse token sera utilizado posteriormente para autorizar as requisicoes ao proxy do modelo de linguagem.


In [ ]:
token = client.login(
    registration_number=userdata.get("PGL_REGISTRATION_NUMBER"),
    password=userdata.get("PGL_PASSWORD"),
)

## 4. Visualizacao do token

A celula abaixo apenas exibe o conteudo do token gerado na etapa anterior. O token e uma string no formato JWT (JSON Web Token), composta por cabecalho, payload e assinatura, separados por pontos.

Atencao: o token deve ser tratado como uma credencial sensivel e nao deve ser compartilhado publicamente.


In [ ]:
token

## 5. Uso do token com o modelo de linguagem

Por fim, o token obtido e utilizado como `api_key` para instanciar um objeto `ChatOpenAI` do `langchain-openai`, configurado para se comunicar com um proxy compativel com a API da OpenAI (`base_url`).

Em seguida, o modelo `gpt-4o-mini` e invocado com uma mensagem simples, e o conteudo da resposta e exibido em tela.


In [ ]:
llm = ChatOpenAI(
    base_url="https://pgl-proxy.vercel.app/v1",
    api_key=token,
    model="gpt-4o-mini",
)

print(llm.invoke("Diga oi").content)